In [6]:
import torch
import torch.nn as nn

In [7]:
class Attention(nn.Module):
    def __init__(self,features,num_heads):
        super().__init__()
        self.num_heads=num_heads
        self.head_features=features//num_heads
        self.q=nn.Linear(features,features)
        self.k=nn.Linear(features,features)
        self.v=nn.Linear(features,features)
    def forward(self,x):
        q=self.q(x)
        k=self.k(x)
        v=self.v(x)
        q=q.view(q.shape[0],q.shape[1],self.num_heads,self.head_features).transpose(1,2)
        k=k.view(k.shape[0],k.shape[1],self.num_heads,self.head_features).transpose(1,2)
        v=v.view(v.shape[0],v.shape[1],self.num_heads,self.head_features).transpose(1,2)
        score=q@k.transpose(-1,-2)
        scaled_score=score/(self.head_features**0.5)
        scaled_score=score.softmax(dim=-1)
        attention=scaled_score@v
        attention=attention.transpose(1,2).contiguous().view(attention.shape[0],attention.shape[2],-1)
        return attention

In [8]:
class FNN(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(features,features*2),
            nn.GELU(),
            nn.Linear(features*2,features)
        )
    def forward(self,x):
        return self.model(x)

In [10]:
features=16
num_heads=2
x=torch.rand((2,10,features),dtype=torch.float32)
attention_model=Attention(features,num_heads)
attention=attention_model(x)
residue=attention+x
norm=nn.LayerNorm(features)
residue=norm(residue)

feed_forward=FNN(features)
print(feed_forward(residue).shape)

torch.Size([2, 10, 16])
